In [1]:
from pathlib import Path
import sys
import time

sys.path.insert(0, str(Path.cwd().resolve().parent.parent))

from ultralytics import YOLO
import torch
import src.config as cfg
from src.training_logger import TrainingLogger

print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

NVIDIA GeForce RTX 4060 Ti


In [2]:
logger = TrainingLogger(log_dir="../../training_logs")

MODEL_SIZE = "s"
EPOCHS = 100
BATCH_SIZE = 16
IMGSZ = 768

In [3]:
model = YOLO(f"yolo26{MODEL_SIZE}.pt")

start = time.time()

results = model.train(
    data=str(cfg.DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH_SIZE,
    patience=20,
    device=0 if torch.cuda.is_available() else "cpu",
)

train_time = time.time() - start

New https://pypi.org/project/ultralytics/8.4.83 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.75  Python-3.12.13 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4060 Ti, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\pedro\OneDrive\Documentos\programao\eletiva-leomar\YOLOCraft\data\minecraft_mobs\minecraft_mobs_yolo\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=768, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max

In [4]:
metrics = model.val(workers=0)

baseline_map50, baseline_map50_95 = 0.9262, 0.7663

print(f"mAP50:    {metrics.box.map50:.4f}  ({metrics.box.map50 - baseline_map50:+.4f})")
print(f"mAP50-95: {metrics.box.map:.4f}  ({metrics.box.map - baseline_map50_95:+.4f})")

Ultralytics 8.4.75  Python-3.12.13 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4060 Ti, 8188MiB)
YOLO26s summary (fused): 122 layers, 9,466,728 parameters, 0 gradients, 20.5 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 1635.8455.8 MB/s, size: 314.9 KB)
val: Scanning C:\Users\pedro\OneDrive\Documentos\programação\eletiva-leomar\YOLOCraft\data\minecraft_mobs\minecraft_mobs_yolo\val\labels.cache... 395 images, 102 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 395/395  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 25/25 3.5it/s 7.1s0.3s
                   all        395        494      0.953      0.922      0.952      0.816
               creeper         74         84      0.937      0.881      0.941      0.823
              skeleton        136        154      0.951      0.909      0.927      0.802
                spider         80         92      0.965      0.989      0.989       0.85
                zombie        1

In [5]:
train_id, _ = logger.log_training(
    model_size=MODEL_SIZE,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    imgsz=IMGSZ,
    map50=metrics.box.map50,
    map50_95=metrics.box.map,
    train_time=train_time,
    notes="Dataset 1",
)

logger.summary()


ID                 Model        Epochs   mAP50      mAP50-95   Time(h)  Date
--------------------------------------------------------------------------------
20260630_184356    yolo26s      100      0.9522     0.8165     2.16     30/06/2026

Best: yolo26s (ID: 20260630_184356) — mAP50: 0.9522
